# Contrato de variables sin PERSONAS

Objetivo: verificar el orden procesado y la exclusión de la familia proxy.

In [1]:
from pathlib import Path
import json
ROOT=Path.cwd()
schema=json.loads((ROOT/'models/final/feature_schema.json').read_text())
features=schema['processed_feature_order']
forbidden={'n_personas','n_pasajeros','n_peatones','n_conductor_fugado','edad_media_involucrados','edad_faltante'}
assert forbidden.isdisjoint(features)
assert schema['processed_feature_count']==len(features)
{'raw_fields':len(schema['required_raw_fields']),'processed_features':len(features),'forbidden_present':sorted(forbidden & set(features))}

{'raw_fields': 21, 'processed_features': 169, 'forbidden_present': []}

## Lectura

El contrato es ejecutable: el orden, ancho y campos crudos están persistidos. PERSONAS no está escondida tras una transformación.

In [2]:
import pandas as pd
audit=pd.read_csv(ROOT/'report/tables/feature_availability_audit.csv')
assert (audit.loc[audit.source_fields.str.contains('PERSONAS'),'decision']=='exclude entire source from predictors').all()
audit

,source_fields,source_stage,timestamp_evidence,target_dependency,allowed_scope,decision
0,"FECHA, HORA",Consolidated registry,timestamp not supplied,not evaluated,retrospective historical classification,Calendar and cyclic-time features
1,"DEPARTAMENTO, coordenadas, red/tipo de vía",Consolidated registry,timestamp not supplied,not evaluated,retrospective historical classification,Location and road context
2,"ZONA, CLIMA, geometría, superficie",Consolidated registry,timestamp not supplied,not evaluated,retrospective historical classification,Scene and infrastructure context
3,CLASE,Consolidated registry,timestamp not supplied,no,retrospective historical classification,Retrospective classification
4,VEHICULOS involucrados (conteos y tipo),VEHICULOS companion registry,timestamp not supplied,not evaluated,retrospective historical classification,include with scope restriction
5,Todos los agregados de PERSONAS,PERSONAS companion registry,not usable: cardinality/deceased counts encode...,yes,none,exclude entire source from predictors
6,"FALLECIDOS, LESIONADOS, VEHICULOS_DANADOS",Outcome/count after event,no,no,retrospective historical classification,Excluded: direct outcome leakage
7,"CAUSA_FACTOR, CAUSA_ESPECIFICA",Investigation conclusion,no,no,retrospective historical classification,Excluded: post-investigation leakage
8,"SENAL_VERTICAL, SENAL_HORIZONTAL",Sparse recording field,not reliable,not reliable,retrospective historical classification,Excluded: missingness is period-dependent


## Disponibilidad

VEHICULOS solo se admite bajo alcance retrospectivo histórico. La fuente no prueba disponibilidad al instante de notificación.

### Data Analysis Key Findings
- PERSONAS está excluida en crudo y procesado.
- La disponibilidad temporal se registra como evidencia ausente, no como supuesto.

### Insights or Next Steps
- No ampliar el alcance a tiempo real sin timestamps por campo.